# 法規資料庫驗證 Notebook
檢視 SQLite + LanceDB 資料正確性

In [ ]:
import sqlite3, pandas as pd, lancedb, os
from sentence_transformers import SentenceTransformer

BASE  = os.path.dirname(os.path.abspath('.'))
DB    = 'land.db'
LANCE = 'regulations_lancedb'

conn = sqlite3.connect(DB)
db   = lancedb.connect(LANCE)
tbl  = db.open_table('regulations')
print('SQLite + LanceDB 載入完成')

## 1. SQLite 資料庫總覽

In [ ]:
# 各資料表筆數
for table in ['land_types','use_items','use_matrix','eval_conditions']:
    n = pd.read_sql(f'SELECT COUNT(*) as n FROM {table}', conn).iloc[0,0]
    print(f'  {table}: {n} 筆')

In [ ]:
# 用地對應總表
df_matrix = pd.read_sql('''
    SELECT m.type_code, lt.type_name, m.item_no, ui.item_name, m.eval_code
    FROM use_matrix m
    JOIN land_types lt ON m.type_code = lt.type_code
    JOIN use_items  ui ON m.item_no   = ui.item_no
    ORDER BY m.type_code, m.item_no
''', conn)
df_matrix

In [ ]:
# 各評估代號條件數量
df_cond = pd.read_sql('''
    SELECT eval_code,
           COUNT(*) as 總條件,
           SUM(CASE WHEN condition_type="hard" THEN 1 ELSE 0 END) as 硬性,
           SUM(CASE WHEN condition_type!="hard" THEN 1 ELSE 0 END) as 加權
    FROM eval_conditions GROUP BY eval_code
''', conn)
df_cond

In [ ]:
# 指定查看某個評估代號的所有條件
EVAL_CODE = 'EE-1'   # ← 改這裡

df = pd.read_sql(f'''
    SELECT condition_no, condition_name, condition_type, note,
           substr(threshold,1,100) as threshold_preview
    FROM eval_conditions
    WHERE eval_code = "{EVAL_CODE}"
    ORDER BY condition_no
''', conn)
df

## 2. LanceDB 法規向量庫

In [ ]:
# 所有法規資料（不含向量）
df_regs = tbl.to_pandas().drop(columns=['vector'])
print(f'總筆數：{len(df_regs)}')
print(df_regs['source'].value_counts())
df_regs.head(10)

In [ ]:
# 依來源篩選
SOURCE = 'PDF'   # 'PDF' | 'eval_conditions' | 'EE總表'
df_regs[df_regs['source'] == SOURCE][['condition','article_no','content']].head(20)

## 3. 向量搜尋測試

In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print('模型載入完成')

In [ ]:
# ← 改這裡輸入你的問題
QUERY = '農牧用地申請再生能源需要什麼條件'
TOP_K = 5

vec     = model.encode(QUERY).tolist()
results = tbl.search(vec).limit(TOP_K).to_list()

for i, r in enumerate(results):
    score = round(1 - r['_distance'], 3)
    print(f'\n─── 第{i+1}筆 (相似度: {score}) ───')
    print(f'來源：{r["source"]} | 代號：{r["eval_code"]} | 條號：{r["article_no"]}')
    print(f'條件：{r["condition"]}')
    print(f'內容：{r["content"][:300]}')

## 4. 資料正確性快速驗證

In [ ]:
# 檢查 use_matrix 所有 eval_code 是否有對應的 eval_conditions
matrix_codes = set(pd.read_sql('SELECT DISTINCT eval_code FROM use_matrix', conn)['eval_code'])
cond_codes   = set(pd.read_sql('SELECT DISTINCT eval_code FROM eval_conditions', conn)['eval_code'])

missing = matrix_codes - cond_codes
print(f'use_matrix 有 {len(matrix_codes)} 個 eval_code')
print(f'eval_conditions 已有 {len(cond_codes)} 個 eval_code')
print(f'缺少條件資料的代號（待補）：{sorted(missing)}')

In [ ]:
# 檢查 regulation_ids 連結
linked = pd.read_sql('''
    SELECT eval_code, COUNT(*) as total,
           SUM(CASE WHEN regulation_ids IS NOT NULL THEN 1 ELSE 0 END) as linked
    FROM eval_conditions GROUP BY eval_code
''', conn)
linked['連結率'] = (linked['linked'] / linked['total'] * 100).round(1).astype(str) + '%'
linked